# L7 artery flow (physics v3): setup, solve and export in Fluent 26.1

Simple, linear version of `artery_pyfluent.ipynb`: one job per cell, direct `solver.settings` paths, Fluent pinned to 26.1, exact zone names. Physics comes only from `case_setup_L7_v3.csv`.

| | Value (from the CSV) | Source |
|---|---|---|
| Lumen | L7: R0 1.22 mm, 73 % area stenosis, full 360° vessel (blocked L7, as in 08_cfd), axis = +z through the origin | 19_CFD_SETTINGS_vishesh.md |
| Blood | 1050 kg/m³, Carreau (μ₀ 0.056, μ∞ 0.00345 Pa·s, λ 3.313 s, n 0.3568) | Cho & Kensey 1991 |
| Inlet | Steady parabolic, 0.10 m/s (rest) or 0.26 m/s (hyperaemia), one per run | Kaufmann 2005 |
| Outlet / wall | 0 Pa gauge / no-slip | 19_CFD_SETTINGS |
| Flow model | laminar for now; laminar or standard k-ω SST (Decision 7 open) | |
| Numerics | Coupled, least-squares gradients, 2nd-order pressure, 2nd-order upwind, residuals 1e-5 | 19_CFD_SETTINGS |

**How to run**
1. Set the run parameters in cell 0 (`INLET_U = 0.10` first).
2. Run every cell from the top. Cells 1 to 4 need no Fluent.
3. For the second velocity: set `INLET_U = 0.26` in cell 0, **restart the kernel and run all** again. Each run writes its own folder.
4. The last cell closes Fluent.

Phase 1 (2 Oct): convergence = residuals 1e-5 AND Δp flat within 0.05 Pa over 20 iterations; segment Δp over the 25 mm artery; face-value, area-weighted WSS on the artery wall over the segment; a comparison with Vishesh's `L7_cfd_qoi.csv` (Δp within 1 %, mean WSS within 2 %).

Phase 2 (2 Oct): `MESH_FROM = "step"` meshes the lumen STEP with Vishesh's watertight workflow and sizing rule, names the zones his way, then solves as above. `MESH_FROM = "case"` reads a finished case (the Phase 1 route).

Not in this version: the pulsatile v3 UDF inlet and TAWSS / OSI (Phase 3).

## 0. Run parameters
The only cell you edit between runs.

In [ ]:
import os
NB_FILE  = globals().get("__vsc_ipynb_file__")              # set by VS Code; Jupyter already starts in the notebook folder
WORK_DIR = os.path.dirname(NB_FILE) if NB_FILE else os.getcwd()
os.chdir(WORK_DIR)                                           # CSVs, cases/ and outputs are relative to this folder
print("working folder:", WORK_DIR)

MESH_FROM      = "case"                            # "case": read MESH_FILE (Phase 1)   "step": mesh LUMEN_STEP (Phase 2)
REMESH         = False                             # False: reuse the M3 mesh if one exists with the same sizes
LUMEN_STEP     = r"geometry\L7_lumen_blocked_full360.step"   # used when MESH_FROM = "step"; the .stl beside it gives the areas
MESH_FILE      = r"cases\L7_blocked_U0p10.cas.h5"   # used when MESH_FROM = "case"; set by cell M3 when MESH_FROM = "step"
SETUP_CSV      = "case_setup_L7_v3.csv"            # full 360 (blocked); case_setup_L7_v3_quarter.csv for a quarter mesh (stented)
FLUENT_VERSION = "26.1.0"                      # pinned (Decision 1)
PROCS          = 4                             # set 1 if the parallel launch times out
SHOW_GUI       = False
OUT_DIR        = "artery_runs_v3"
INLET_U        = 0.10                          # m/s: 0.10 rest or 0.26 hyperaemia; restart and run all for each
QOI_FILE       = "L7_cfd_qoi.csv"                  # Vishesh's pipeline results (08_cfd, 27 Sep)
QOI_CASE       = "L7_blocked"                      # L7_blocked or L7_stented; the row is picked with INLET_U

## 1. Imports

In [ ]:
import datetime
import json
import math
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Read and check the setup CSV
Every key has a parser. An unknown, duplicate or missing key stops here, before Fluent starts.

In [ ]:
def f_(v):  return float(v)
def i_(v):  return int(float(v))
def s_(v):  return str(v).strip()
def yes_(v): return str(v).strip().lower() in ("yes", "y", "true", "1")
def list_(v): return [p.strip() for p in str(v).split(";") if p.strip()]
def flist_(v): return [float(p) for p in list_(v)]
def vec_(v):
    out = flist_(v)
    if len(out) != 3:
        raise ValueError(f"expected 3 components separated by ';', got {v!r}")
    return out
def one_of(*opts):
    def p(v):
        v = str(v).strip().lower()
        if v not in opts:
            raise ValueError(f"{v!r} is not one of {opts}")
        return v
    return p

# key: (parser, required)
SPEC = {
    "density": (f_, True), "viscosity_model": (one_of("carreau", "newtonian"), True),
    "carreau_mu0": (f_, True), "carreau_mu_inf": (f_, True),
    "carreau_lambda": (f_, True), "carreau_n": (f_, True),
    "mesh_units": (one_of("m", "mm"), True), "R0": (f_, True), "sector_fraction": (f_, True),
    "axis_origin": (vec_, True), "axis_direction": (vec_, True), "stenosis_area": (f_, True),
    "zone_inlet": (list_, True), "zone_outlet": (list_, True), "zone_wall": (list_, True),
    "zone_symmetry": (list_, False),
    "inlet_mean_velocity": (flist_, True), "inlet_profile": (one_of("parabolic", "uniform", "udf"), True),
    "outlet_gauge_pressure": (f_, True),
    "viscous_model": (one_of("laminar", "sst"), True),
    "inlet_turb_intensity": (f_, True), "inlet_hydraulic_diameter": (f_, True),
    "time": (one_of("steady", "transient"), True),
    "p_v_coupling": (s_, True), "gradient": (s_, True),
    "disc_pressure": (s_, True), "disc_convective": (s_, True),
    "residual_target": (f_, True), "max_iterations": (i_, True),
    "dp_flat_window": (i_, True), "dp_flat_tol": (f_, True),
    "stability_tol": (f_, True), "stability_window": (f_, True), "mass_imbalance_tol": (f_, True),
    "inlet_area_tol": (f_, True), "inlet_mass_flow_tol": (f_, True),
    "wss_sampling": (one_of("face", "node"), True), "low_wss_thresholds": (flist_, True),
    "segment_z_start": (f_, True), "segment_length": (f_, True),
    "bc_stage": (s_, False), "export_field": (yes_, False),
}

raw = pd.read_csv(SETUP_CSV, comment="#", dtype=str, keep_default_na=False)
raw["key"] = raw["key"].str.strip()
raw = raw[raw["key"] != ""]
problems = []
dups = raw["key"][raw["key"].duplicated()].tolist()
if dups:
    problems.append(f"duplicate keys: {dups}")
unknown = [k for k in raw["key"] if k not in SPEC]
if unknown:
    problems.append(f"unknown keys: {unknown}")
cfg = {}
for k, v in zip(raw["key"], raw["value"]):
    if k not in SPEC:
        continue
    parser, _ = SPEC[k]
    if str(v).strip() == "":
        cfg[k] = [] if parser in (list_, flist_) else None
        continue
    try:
        cfg[k] = parser(v)
    except Exception as e:
        problems.append(f"{k} = {v!r}: {e}")
missing = [k for k, (_, req) in SPEC.items() if req and cfg.get(k) in (None, [])]
if missing:
    problems.append(f"missing required keys: {missing}")
if problems:
    raise ValueError("case_setup problems:\n  " + "\n  ".join(problems))

cfg.setdefault("zone_symmetry", [])
cfg.setdefault("export_field", False)
if cfg["time"] != "steady":
    raise NotImplementedError("time = transient comes with the v3 UDF inlet in Phase 3")
if cfg["axis_origin"] != [0.0, 0.0, 0.0] or cfg["axis_direction"] != [0.0, 0.0, 1.0]:
    raise ValueError("this notebook assumes the pipeline frame: axis = +z through the origin")
if cfg["sector_fraction"] < 1 and not cfg["zone_symmetry"]:
    raise ValueError("sector_fraction < 1 but zone_symmetry is empty")
if cfg["wss_sampling"] != "face":
    raise NotImplementedError("wss_sampling = node is not wired; Decision 4 is face values, area-weighted")
if len(cfg["zone_inlet"]) != 1 or len(cfg["zone_outlet"]) != 1:
    raise ValueError("exactly one inlet zone and one outlet zone are expected")

ZONE_INLET, ZONE_OUTLET = cfg["zone_inlet"][0], cfg["zone_outlet"][0]
ZONE_WALL, ZONE_SYM = cfg["zone_wall"], cfg["zone_symmetry"]
print(f"viscous model: {cfg['viscous_model']}   inlet: {cfg['inlet_profile']}   "
      f"velocities in the CSV: {cfg['inlet_mean_velocity']} m/s")
raw[["key", "value", "units", "source"]]

## 3. Flow numbers
Inlet flow and Reynolds numbers for the velocity in cell 0. Cell 9 calls this again, so a re-run for the second velocity updates them.

In [ ]:
def carreau(gdot):
    return cfg["carreau_mu_inf"] + (cfg["carreau_mu0"] - cfg["carreau_mu_inf"]) * \
        (1 + (cfg["carreau_lambda"] * gdot) ** 2) ** ((cfg["carreau_n"] - 1) / 2)

def flow_numbers(U):
    if not any(abs(U - u) < 1e-9 for u in cfg["inlet_mean_velocity"]):
        raise ValueError(f"INLET_U = {U} is not in inlet_mean_velocity {cfg['inlet_mean_velocity']}")
    R0, rho = cfg["R0"], cfg["density"]
    A_nom = cfg["sector_fraction"] * math.pi * R0 ** 2
    gdot_w = 4 * U / R0                                   # Poiseuille wall shear rate
    mu_w = carreau(gdot_w) if cfg["viscosity_model"] == "carreau" else cfg["carreau_mu_inf"]
    return dict(
        U=U, A_inlet_nominal_m2=A_nom,
        Q_full_vessel_mL_min=U * math.pi * R0 ** 2 * 6e7,
        Re_mu_inf=rho * U * 2 * R0 / cfg["carreau_mu_inf"],
        Re_carreau_wall=rho * U * 2 * R0 / mu_w,
        throat_radius_mm=R0 * math.sqrt(1 - cfg["stenosis_area"]) * 1e3,
        U_throat_mean=U / (1 - cfg["stenosis_area"]),
    )

FLOW = flow_numbers(INLET_U)
for k, v in FLOW.items():
    print(f"  {k:22s} {v:.6g}")

## M0. Vishesh's pipeline modules (Phase 2)
Imports his mesher, zone tools and launcher from `23_CFD_PIPELINE_v3/StenosisCFD`, unchanged. His launcher is told to use this notebook's Fluent version and process count. All M cells do nothing when `MESH_FROM = "case"`.

In [ ]:
import sys
if MESH_FROM not in ("case", "step"):
    raise ValueError(f"MESH_FROM must be 'case' or 'step', got {MESH_FROM!r}")
PIPE_DIR = (Path.cwd().parent / "23_CFD_PIPELINE_v3" / "StenosisCFD").resolve()
if MESH_FROM == "step":
    os.environ["STENOSIS_FLUENT_PRODUCT_VERSION"] = FLUENT_VERSION     # read by his launcher at import
    os.environ["STENOSIS_FLUENT_NPROC"] = str(PROCS)
    if str(PIPE_DIR) not in sys.path:
        sys.path.insert(0, str(PIPE_DIR))
    import cfd_config_vishesh as VC          # MeshControls
    import cfd_mesh_vishesh as VM            # watertight workflow tasks, mesh report, write
    import cfd_run_vishesh as VR             # zone split / centroid / validate, reports
    import fluent_launch_vishesh as VL       # launcher with the licence-seat retry
    print(f"pipeline modules from {PIPE_DIR}; launcher: Fluent {VL.PRODUCT_VERSION}, {VL.PROCESSOR_COUNT} process(es)")

## M1. Lumen geometry (no Fluent)
Reads the STL exported beside the STEP: length, inlet and throat radius, quarter or full, and the area of each face family (the expected areas for the zone check in Z1). Stops if the lumen disagrees with the CSV.

In [ ]:
def read_stl(path):
    b = Path(path).read_bytes()
    n = int(np.frombuffer(b, "<u4", 1, 80)[0])
    if 84 + 50 * n != len(b):
        raise ValueError(f"{path}: expected a binary STL")
    rec = np.frombuffer(b, np.dtype([("n", "<f4", 3), ("v", "<f4", (3, 3)), ("a", "<u2")]), n, 84)
    return rec["v"].astype(float)

if MESH_FROM == "step":
    STEP_PATH = Path(LUMEN_STEP).resolve()
    STL_PATH = STEP_PATH.with_suffix(".stl")
    for p in (STEP_PATH, STL_PATH):
        if not p.exists():
            raise FileNotFoundError(p)
    T = read_stl(STL_PATH)                                    # mm
    P = T.reshape(-1, 3)
    L_mm = float(P[:, 2].max())
    tol = 1e-6 * L_mm
    tri_area = 0.5 * np.linalg.norm(np.cross(T[:, 1] - T[:, 0], T[:, 2] - T[:, 0]), axis=1)
    fam = np.full(len(T), "wall", dtype=object)
    fam[np.all(np.abs(T[:, :, 2]) < tol, 1)] = "inlet"
    fam[np.all(np.abs(T[:, :, 2] - L_mm) < tol, 1)] = "outlet"
    QUARTER = bool(P[:, 0].min() > -tol and P[:, 1].min() > -tol)
    if QUARTER:
        fam[np.all(np.abs(T[:, :, 0]) < tol, 1)] = "symmetry"
        fam[np.all(np.abs(T[:, :, 1]) < tol, 1)] = "symmetry"
    wall_pts = T[fam == "wall"].reshape(-1, 3)
    r_wall = np.hypot(wall_pts[:, 0], wall_pts[:, 1])
    inlet_pts = P[np.abs(P[:, 2]) < tol]
    GEOM = dict(
        length_mm=L_mm, quarter=QUARTER,
        inlet_radius_mm=float(np.hypot(inlet_pts[:, 0], inlet_pts[:, 1]).max()),
        throat_radius_mm=float(r_wall.min()), throat_z_mm=float(wall_pts[r_wall.argmin(), 2]),
        expected_mm2={"inlet": float(tri_area[fam == "inlet"].sum()), "outlet": float(tri_area[fam == "outlet"].sum()),
                      "wall": float(tri_area[fam == "wall"].sum()),
                      **({"symmetry_each": float(tri_area[fam == "symmetry"].sum() / 2)} if QUARTER else {})},
    )
    problems = []
    if QUARTER != (cfg["sector_fraction"] < 1):
        problems.append(f"lumen is {'quarter' if QUARTER else 'full 360'} but sector_fraction = {cfg['sector_fraction']}: use the matching CSV")
    if abs(GEOM["inlet_radius_mm"] / (cfg["R0"] * 1e3) - 1) > 0.005:
        problems.append(f"inlet radius {GEOM['inlet_radius_mm']:.4f} mm vs R0 {cfg['R0'] * 1e3:.4f} mm")
    seg0, seg1 = cfg["segment_z_start"] * 1e3, (cfg["segment_z_start"] + cfg["segment_length"]) * 1e3
    if not (seg0 < GEOM["throat_z_mm"] < seg1) or L_mm < seg1:
        problems.append(f"throat at z = {GEOM['throat_z_mm']:.2f} mm or length {L_mm:.1f} mm does not fit the segment {seg0:.1f}-{seg1:.1f} mm")
    if problems:
        raise ValueError("lumen and CSV disagree:\n  " + "\n  ".join(problems))
    print(json.dumps(GEOM, indent=2))

## M2. Mesh sizes (Vishesh's throat-adapted rule)
Surface max size = throat radius ÷ `cells_per_throat_radius`, clamped to [floor, cap]; min = max ÷ `max_to_min_ratio`; growth, curvature angle and prism layers from the same block of `cfd_pipeline_v3_vishesh.yaml`. Port of `cfd_v3_config_vishesh.mesh_controls_v3` (Vishesh Kumar), because that module imports code not in the repo. A mesh-study (H2) result changes the yaml, not this cell. Needs `pip install pyyaml`.

In [ ]:
if MESH_FROM == "step":
    import yaml
    MESH_YAML = PIPE_DIR / "config" / "cfd_pipeline_v3_vishesh.yaml"
    mb = yaml.safe_load(MESH_YAML.read_text())["mesh"]
    if mb["rule"] != "throat_adapted":
        raise ValueError(f"mesh.rule {mb['rule']!r}: only throat_adapted is implemented")
    rt = GEOM["throat_radius_mm"]
    max_mm = min(float(mb["max_size_cap_mm"]), max(float(mb["max_size_floor_mm"]), rt / float(mb["cells_per_throat_radius"])))
    min_mm = max_mm / float(mb["max_to_min_ratio"])
    MESH_CTRL = VC.MeshControls(
        min_size_mm=round(min_mm, 4), max_size_mm=round(max_mm, 4),
        growth_rate=float(mb["growth_rate"]), curvature_normal_angle=float(mb["curvature_normal_angle_deg"]),
        n_prism_layers=int(mb["prism_layers"]), prism_transition_ratio=float(mb["prism_transition_ratio"]),
        prism_growth_rate=float(mb["prism_growth_rate"]), volume_fill=str(mb["volume_fill"]))
    print(f"throat radius {rt:.4f} mm -> surface max {MESH_CTRL.max_size_mm} mm, min {MESH_CTRL.min_size_mm} mm, "
          f"{MESH_CTRL.n_prism_layers} prism layers, {MESH_CTRL.volume_fill}")

## M3. Mesh in Fluent Meshing (Vishesh's watertight workflow)
His task functions, called in his order. The import asks for one zone per CAD face (port of `cfd_v2_vishesh._import_geometry_v2`: with one zone per body the extended lumen lost its inlet, measured 24 Sep); if Fluent refuses it, the cell stops rather than falling back.

In [ ]:
if MESH_FROM == "step":
    MESH_DIR = Path(OUT_DIR) / "meshes"
    MESH_DIR.mkdir(parents=True, exist_ok=True)
    MESH_OUT = (MESH_DIR / f"{STEP_PATH.stem}_v3.msh.h5").resolve()
    REPORT_PATH = MESH_DIR / f"{STEP_PATH.stem}_mesh_report.json"
    SIZES = [MESH_CTRL.min_size_mm, MESH_CTRL.max_size_mm]
    old = json.loads(REPORT_PATH.read_text()) if REPORT_PATH.exists() else None
    REUSE = (not REMESH) and MESH_OUT.exists() and old is not None and old.get("sizes_mm") == SIZES
if MESH_FROM == "step" and REUSE:
    MESH_REPORT = old
    print(f"reusing {MESH_OUT.name} (same sizes {SIZES} mm; set REMESH = True to mesh again)")
elif MESH_FROM == "step":
    t0 = time.time()
    meshing = VL.launch(mode="meshing", cwd=str(MESH_DIR.resolve()))
    try:
        meshing.workflow.InitializeWorkflow(WorkflowType="Watertight Geometry")
        task = VM._task(meshing, "Import Geometry")
        task.Arguments = dict(FileName=str(STEP_PATH), LengthUnit="mm", AppendMesh=False,
                              Directory=str(STEP_PATH.parent), CadImportOptions=dict(OneZonePer="face"))
        task.Execute()
        VM.add_local_sizing(meshing)
        VM.generate_surface_mesh(meshing, MESH_CTRL)
        VM.describe_geometry(meshing)
        VM.add_boundary_layers(meshing, MESH_CTRL)
        VM.generate_volume_mesh(meshing, MESH_CTRL)
        MESH_REPORT = VM.mesh_report(meshing)
        VM.write_mesh(meshing, str(MESH_OUT))
    finally:
        meshing.exit()
    MESH_REPORT.update(mesh_path=str(MESH_OUT), mesh_walltime_s=round(time.time() - t0, 1),
                       sizes_mm=SIZES, geometry=GEOM)
    print(json.dumps({k: v for k, v in MESH_REPORT.items() if k != "geometry"}, indent=2, default=str))

## M4. Mesh gates
Stops if the minimum orthogonal quality is below the target (0.1) or the cell count reaches the Student cap (512,000). The mesh written in M3 becomes `MESH_FILE`.

In [ ]:
if MESH_FROM == "step":
    cells, oq = MESH_REPORT.get("cell_count"), MESH_REPORT.get("min_orthogonal_quality")
    if cells is None or oq is None:
        raise RuntimeError(f"mesh report incomplete: {MESH_REPORT}")
    if oq < VC.MeshControls().min_orthogonal_quality_target:
        raise RuntimeError(f"min orthogonal quality {oq:.3f} < {VC.MeshControls().min_orthogonal_quality_target}")
    if cells >= VC.MeshControls().student_cell_cap:
        raise RuntimeError(f"{cells} cells >= Student cap {VC.MeshControls().student_cell_cap}")
    REPORT_PATH.write_text(json.dumps(MESH_REPORT, indent=2, default=str))
    MESH_FILE = str(MESH_OUT)
    print(f"[mesh gates] {cells} cells, min orthogonal quality {oq:.3f}: OK -> {MESH_FILE}")

## 4. Check the mesh file (no Fluent)
A gmsh-native `.msh` has the same extension as a Fluent mesh but Fluent cannot read it; this catches it here.

In [ ]:
MESH_PATH = Path(MESH_FILE)
if not MESH_PATH.exists():
    raise FileNotFoundError(MESH_PATH)
name = MESH_PATH.name.lower()
if name.endswith((".cas", ".cas.h5", ".cas.gz")):
    MESH_KIND = "case"
elif name.endswith((".msh", ".msh.h5", ".msh.gz")):
    MESH_KIND = "mesh"
else:
    raise ValueError(f"{MESH_PATH.name}: expected .cas.h5 / .cas / .msh.h5 / .msh")
if name.endswith(".msh") and MESH_PATH.open("rb").read(64).lstrip().startswith(b"$MeshFormat"):
    raise ValueError(f"{MESH_PATH.name} is a gmsh-native mesh, not a Fluent mesh")
MESH_STEM = MESH_PATH.name.split(".")[0]
print(f"{MESH_KIND}: {MESH_PATH.resolve()}  ({MESH_PATH.stat().st_size / 1e6:.2f} MB)")

## 5. Launch Fluent 26.1
Same launch arguments as Vishesh's `fluent_launch_vishesh.py`.

In [ ]:
import ansys.fluent.core as pyfluent

out = Path(OUT_DIR)
out.mkdir(parents=True, exist_ok=True)
if MESH_FROM == "step":
    solver = VL.launch_solver_with_retry(str(out.resolve()))   # waits out the meshing session's licence release
else:
    solver = pyfluent.launch_fluent(
    product_version=FLUENT_VERSION,
        precision="double",
        dimension=3,
        mode="solver",
        processor_count=PROCS,
        ui_mode="gui" if SHOW_GUI else "no_gui",
        start_timeout=300,
        cwd=str(out.resolve()),
    )
s = solver.settings
print(solver.get_fluent_version(), "| pyfluent", pyfluent.__version__)

## 6. Read the case or mesh
Stops if a zone named in the CSV is missing or has the wrong type. Exact names; no wildcards, no re-typing.

In [ ]:
s.file.read(file_type=MESH_KIND, file_name=str(MESH_PATH.resolve()))
if cfg["mesh_units"] == "mm":
    s.mesh.scale(x_scale=1e-3, y_scale=1e-3, z_scale=1e-3)
s.setup.general.solver.type = "pressure-based"
s.setup.general.solver.time = "steady"
s.setup.models.energy.enabled = False
s.mesh.check()

## Z1. Name the zones (Vishesh's method, Phase 2)
A meshed STEP arrives with one generic wall zone per CAD face. If the zone count differs from the CAD faces (3 full, 5 quarter), split at 40° (his `separate_boundary_zones`); classify each by its centroid (his rule; the full-360 variant skips the symmetry test), validate (his `validate_zone_map`), check every zone's area against the STL within 3 % with his 27 Sep cap waiver, then retype and rename to `inlet` / `outlet` / `wall` / `sym-1` / `sym-2`. Ports of `cfd_v2_vishesh.check_zone_areas`, `cap_waiver` and `_classify_full360` (Vishesh Kumar).

In [ ]:
def zone_family(c, L_m, R0_m, quarter):
    xc, yc, zc = c
    if abs(zc) <= 0.02 * L_m:
        return "inlet"
    if abs(zc - L_m) <= 0.02 * L_m:
        return "outlet"
    if quarter and (abs(yc) <= 0.05 * R0_m or abs(xc) <= 0.05 * R0_m):
        return "symmetry"
    return "wall"

def zone_area_m2(z):
    return abs(float(solver.fields.reduction.area(locations=[z])))   # Vishesh's route (cfd_v2_vishesh._zone_area_m2); reports are inactive here

if MESH_FROM == "step":
    pipe = __import__("yaml").safe_load(MESH_YAML.read_text())["zone_area"]
    n_faces = 5 if GEOM["quarter"] else 3                # CAD faces: inlet, outlet, wall (+ 2 symmetry planes)
    n_zones = len(VR._zones_by_type(solver))
    if n_zones == n_faces:
        print(f"{n_zones} zones = {n_faces} CAD faces (one zone per face): no split needed")
    else:
        print(f"{n_zones} zones for {n_faces} CAD faces: splitting at 40 deg (Vishesh's separate_boundary_zones)")
        VR.separate_boundary_zones(solver)
    L_m, R0_m, Q = GEOM["length_mm"] * 1e-3, cfg["R0"], GEOM["quarter"]
    ZMAP = {}
    for z in VR._zones_by_type(solver):
        c = VR._zone_centroid(solver, z)
        if c is None:
            raise RuntimeError(f"no centroid for zone {z}")
        ZMAP[z] = zone_family(c, L_m, R0_m, Q)
    ok, why = VR.validate_zone_map(ZMAP, Q)
    if not ok:
        raise RuntimeError(f"zone classification failed: {why}; {ZMAP}")

    exp = GEOM["expected_mm2"]
    table, problems = {}, []
    for z, f in ZMAP.items():
        a = zone_area_m2(z) * 1e6
        e = exp["symmetry_each" if f == "symmetry" else f]
        table[z] = dict(family=f, area_mm2=a, expected_mm2=e, rel_err=(a - e) / e)
        if abs(table[z]["rel_err"]) > 0.03:
            problems.append(f"{z} classified {f}: area {a:.4f} mm2 vs expected {e:.4f} ({100 * table[z]['rel_err']:+.1f} %)")
    caps = {r["family"]: r["area_mm2"] for r in table.values() if r["family"] in ("inlet", "outlet")}
    waived = []
    for pr in list(problems):                 # 27 Sep cap waiver: caps agree within 0.5 % and sit within 10 % of expected
        if (" classified inlet:" in pr or " classified outlet:" in pr) and len(caps) == 2 \
                and abs(caps["inlet"] - caps["outlet"]) <= pipe["cap_pair_agreement"] * caps["inlet"] \
                and abs(caps["inlet"] - exp["inlet"]) <= pipe["cap_vs_expected"] * exp["inlet"]:
            problems.remove(pr)
            waived.append(pr)
    print(pd.DataFrame(table).T.to_string())
    if waived:
        print("[cap waiver]", waived)
    if problems:
        raise RuntimeError("zone-area check failed (a zone is mis-classified):\n  " + "\n  ".join(problems))

    sym_n = 0
    for z, f in ZMAP.items():                 # retype, then rename (cfd_run_vishesh.setup_boundaries, rename part)
        ftype = {"inlet": "velocity-inlet", "outlet": "pressure-outlet", "symmetry": "symmetry", "wall": "wall"}[f]
        solver.tui.mesh.modify_zones.zone_type(z, ftype)
        if f == "symmetry":
            sym_n += 1
            new = f"sym-{sym_n}"
        else:
            new = f
        if new != z:
            solver.tui.mesh.modify_zones.zone_name(z, new)
    ZONE_TABLE = table
    print("zones renamed:", {z: f for z, f in ZMAP.items()})

## 6b. Check the zones
Stops if a zone named in the CSV is missing or has the wrong type. Exact names; no wildcards.

In [ ]:
bc = s.setup.boundary_conditions
zones = {}
for ztype in ("velocity_inlet", "pressure_outlet", "wall", "symmetry", "interior"):
    for zname in getattr(bc, ztype).get_object_names():
        zones[zname] = ztype

expected = {ZONE_INLET: "velocity_inlet", ZONE_OUTLET: "pressure_outlet"}
expected.update({z: "wall" for z in ZONE_WALL})
expected.update({z: "symmetry" for z in ZONE_SYM})
wrong = [f"{z}: expected {t}, found {zones.get(z, 'missing')}" for z, t in expected.items() if zones.get(z) != t]
if wrong:
    raise ValueError("zone check failed:\n  " + "\n  ".join(wrong) + f"\nzones in the case: {zones}")
unused = [z for z, t in zones.items() if t != "interior" and z not in expected]
if unused:
    print(f"[warn] boundary zones not named in the CSV (left as they are): {unused}")

FLUID_ZONES = list(s.setup.cell_zone_conditions.fluid.get_object_names())
if not FLUID_ZONES:
    raise RuntimeError("no fluid cell zone in the case")
print("fluid cell zones:", FLUID_ZONES)
pd.DataFrame(sorted(zones.items()), columns=["zone", "type"])

## 7. Viscous model
Laminar, or standard k-ω SST (Decision 7 is open; Phase 1 uses laminar).
Set before the material: Fluent offers Carreau only once the flow model is laminar (a fresh mesh starts on its default turbulence model).

In [ ]:
viscous = s.setup.models.viscous
if cfg["viscous_model"] == "laminar":
    viscous.model = "laminar"
else:
    viscous.model = "k-omega"
    viscous.k_omega_model = "sst"
print("viscous model:", viscous.model())

## 8. Material: blood, Carreau
Stops if any constant Fluent holds differs from the CSV.

In [ ]:
fluids = s.setup.materials.fluid
if "blood" not in fluids.get_object_names():
    fluids.create("blood")
blood = fluids["blood"]
blood.density.option = "constant"
blood.density.value = cfg["density"]
if cfg["viscosity_model"] == "carreau":
    blood.viscosity.option = "carreau"
    blood.viscosity.carreau.zero_shear_viscosity = cfg["carreau_mu0"]
    blood.viscosity.carreau.infinite_shear_viscosity = cfg["carreau_mu_inf"]
    blood.viscosity.carreau.time_constant = cfg["carreau_lambda"]
    blood.viscosity.carreau.power_law_index = cfg["carreau_n"]
else:
    blood.viscosity.option = "constant"
    blood.viscosity.value = cfg["carreau_mu_inf"]

for z in FLUID_ZONES:
    s.setup.cell_zone_conditions.fluid[z].material = "blood"     # path checked in Task 0.3

rv = s.setup.reference_values
rv.density = cfg["density"]
rv.viscosity = cfg["carreau_mu_inf"]
rv.length = 2 * cfg["R0"]

print("density:", blood.density.value())
if cfg["viscosity_model"] == "carreau":
    rb = blood.viscosity.carreau.get_state()
    print(json.dumps(rb, indent=2, default=str))       # also shows any temperature term (should be 0 / off)
    want = {"zero_shear_viscosity": cfg["carreau_mu0"], "infinite_shear_viscosity": cfg["carreau_mu_inf"],
            "time_constant": cfg["carreau_lambda"], "power_law_index": cfg["carreau_n"]}
    bad = {k: (rb.get(k), v) for k, v in want.items() if rb.get(k) is None or abs(rb[k] - v) > 1e-6 * abs(v)}
    if bad or abs(blood.density.value() - cfg["density"]) > 1e-6 * cfg["density"]:
        raise RuntimeError(f"blood constants not as set (Fluent, CSV): {bad}")

## 9. Run tag and reports
Each run (one mesh, one `INLET_U`) gets its own folder and monitor file. For another velocity, restart and run all.

In [ ]:
FLOW = flow_numbers(INLET_U)
TAG = f"{MESH_STEM}_{cfg['viscous_model']}_U{INLET_U:.2f}".replace(".", "p")
CASE_DIR = out / TAG
CASE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(SETUP_CSV, CASE_DIR / "case_setup_used.csv")
print(f"==== {TAG} ====")

rd = s.solution.report_definitions
rd.surface["a_inlet"] = {}
rd.surface["a_inlet"].report_type = "surface-area"
rd.surface["a_inlet"].surface_names = [ZONE_INLET]

rd.flux["mflow_inlet"] = {}
rd.flux["mflow_inlet"].report_type = "flux-massflow"
rd.flux["mflow_inlet"].boundaries = [ZONE_INLET]
rd.flux["mflow_outlet"] = {}
rd.flux["mflow_outlet"].report_type = "flux-massflow"
rd.flux["mflow_outlet"].boundaries = [ZONE_OUTLET]

rd.surface["p_inlet"] = {}                       # = pressure drop (outlet is 0 Pa); used for dp-flat in Phase 1
rd.surface["p_inlet"].report_type = "surface-areaavg"
rd.surface["p_inlet"].field = "pressure"
rd.surface["p_inlet"].surface_names = [ZONE_INLET]

rd.surface["wss_wall_areaavg"] = {}
rd.surface["wss_wall_areaavg"].report_type = "surface-areaavg"
rd.surface["wss_wall_areaavg"].field = "wall-shear"
rd.surface["wss_wall_areaavg"].surface_names = ZONE_WALL
rd.surface["wss_wall_max"] = {}
rd.surface["wss_wall_max"].report_type = "surface-facetmax"
rd.surface["wss_wall_max"].field = "wall-shear"
rd.surface["wss_wall_max"].surface_names = ZONE_WALL

rd.volume["vmax"] = {}
rd.volume["vmax"].report_type = "volume-max"
rd.volume["vmax"].field = "velocity-magnitude"
rd.volume["vmax"].cell_zones = FLUID_ZONES

MON_DEFS = ["mflow_inlet", "mflow_outlet", "p_inlet", "wss_wall_areaavg", "wss_wall_max", "vmax"]
MON_FILE = (CASE_DIR / f"{TAG}_monitors.out").resolve()
rf = s.solution.monitor.report_files
rf["monitors"] = {}
rf["monitors"].report_defs = MON_DEFS
rf["monitors"].file_name = str(MON_FILE)
rf["monitors"].frequency = 1

def compute(names):
    vals = {}
    for item in s.solution.report_definitions.compute(report_defs=list(names)) or []:
        for k, v in item.items():
            vals[k] = float(v[0] if isinstance(v, (list, tuple)) else v)
    return vals

def read_monitor_file(path):
    rows, header = [], None
    if not Path(path).exists():
        return None, None
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        if line.startswith("("):
            header = [h.strip('"') for h in line.strip("()").split('" "')]
            continue
        try:
            rows.append([float(v) for v in line.split()])
        except ValueError:
            continue
    return header, (np.array(rows) if rows else None)

## 10. Gate 1: inlet area
Catches a mm/m mix-up and a full/quarter mismatch before anything is solved. The measured radius is recorded in the summary; the inlet profile uses the nominal R0, as the pipeline does.

In [ ]:
s.solution.initialization.standard_initialize()
A_IN = abs(compute(["a_inlet"])["a_inlet"])
ratio = A_IN / FLOW["A_inlet_nominal_m2"]
print(f"[gate 1] inlet area {A_IN:.6e} m2 vs nominal {FLOW['A_inlet_nominal_m2']:.6e} m2 (ratio {ratio:.4f})")
if abs(ratio - 1) > cfg["inlet_area_tol"]:
    hint = "check R0, sector_fraction, mesh_units and zone_inlet"
    if 0.5e6 < ratio < 2e6:
        hint = "the mesh looks like it is in mm: set mesh_units = mm"
    elif 0.5e-6 < ratio < 2e-6:
        hint = "the mesh is scaled but already in m: set mesh_units = m"
    elif abs(ratio * cfg["sector_fraction"] - 1) < 0.05:
        hint = "the mesh looks like a full vessel: set sector_fraction = 1"
    raise RuntimeError(f"inlet area gate failed: {hint}")
R_IN = math.sqrt(A_IN / (cfg["sector_fraction"] * math.pi))

## 11. Boundary conditions
The only cell that sets the inlet.

In [ ]:
if cfg.get("bc_stage"):
    raise NotImplementedError("bc_stage (Harshita's H3 module) is wired in Phase 2, step B1; leave it blank")

inlet = bc.velocity_inlet[ZONE_INLET]
inlet.momentum.velocity_specification_method = "Magnitude, Normal to Boundary"
if cfg["inlet_profile"] == "parabolic":
    # radius = nominal R0 from the CSV: the same expression as the pipeline / 08_cfd cases (Task 0.3, 2 Oct)
    INLET_EXPR = (f"max(0[m/s], 2*{INLET_U:.10g}[m/s]*"
                  f"(1-(Position.x^2+Position.y^2)/({cfg['R0']:.10g}[m])^2))")
    s.setup.named_expressions["u_inlet"] = {"definition": INLET_EXPR}
    inlet.momentum.velocity_magnitude = "u_inlet"
elif cfg["inlet_profile"] == "uniform":
    INLET_EXPR = None
    inlet.momentum.velocity_magnitude = INLET_U
else:
    raise NotImplementedError("inlet_profile = udf: the v3 pulsatile UDF is compiled and hooked here "
                              "from Phase 3 (Decision 8)")
if cfg["viscous_model"] == "sst":
    inlet.turbulence.turbulence_specification = "Intensity and Hydraulic Diameter"   # name checked in Task 0.3
    inlet.turbulence.turbulent_intensity = cfg["inlet_turb_intensity"]
    inlet.turbulence.hydraulic_diameter = cfg["inlet_hydraulic_diameter"]

outlet = bc.pressure_outlet[ZONE_OUTLET]
outlet.momentum.gauge_pressure = cfg["outlet_gauge_pressure"]
if cfg["viscous_model"] == "sst":
    outlet.turbulence.turbulence_specification = "Intensity and Hydraulic Diameter"
    outlet.turbulence.backflow_turbulent_intensity = cfg["inlet_turb_intensity"]
    outlet.turbulence.backflow_hydraulic_diameter = cfg["inlet_hydraulic_diameter"]

s.setup.reference_values.velocity = INLET_U
print("inlet:", INLET_EXPR or f"uniform {INLET_U} m/s")

## 12. Numerics
Residual criterion type `absolute` on Fluent's (default) scaled residuals, as in Vishesh's pipeline.

In [ ]:
methods = s.solution.methods
methods.p_v_coupling.flow_scheme = cfg["p_v_coupling"]
methods.spatial_discretization.gradient_scheme = cfg["gradient"]
methods.spatial_discretization.discretization_scheme["pressure"] = cfg["disc_pressure"]
methods.spatial_discretization.discretization_scheme["mom"] = cfg["disc_convective"]
if cfg["viscous_model"] == "sst":
    methods.spatial_discretization.discretization_scheme["k"] = cfg["disc_convective"]
    methods.spatial_discretization.discretization_scheme["omega"] = cfg["disc_convective"]

residual = s.solution.monitor.residual
residual.options.criterion_type = "absolute"
for eq in residual.equations.get_object_names():
    residual.equations[eq].check_convergence = True
    residual.equations[eq].absolute_criteria = cfg["residual_target"]

print("coupling:", methods.p_v_coupling.flow_scheme())
print("schemes:", methods.spatial_discretization.discretization_scheme.get_state())
print("residual equations:", list(residual.equations.get_object_names()))

## 13. Initialise, Gate 2: inlet mass flow
Catches a wrong axis, radius or units in the parabolic profile. Target = the v3 nominal flow rho·U·sector·π·R0²; tolerance from the CSV (1 %: the faceted inlet delivers +0.72 % on the L7 case, checked 2 Oct).

In [ ]:
s.solution.initialization.hybrid_initialize()
MDOT_TARGET = cfg["density"] * INLET_U * FLOW["A_inlet_nominal_m2"]     # v3 nominal flow (sector * pi * R0^2)
MDOT_INIT = abs(compute(["mflow_inlet"])["mflow_inlet"])
err = abs(MDOT_INIT - MDOT_TARGET) / MDOT_TARGET
print(f"[gate 2] inlet mass flow {MDOT_INIT:.6e} kg/s vs rho*U*pi*R0^2 {MDOT_TARGET:.6e} kg/s ({err:.3%})")
if err > cfg["inlet_mass_flow_tol"]:
    raise RuntimeError("inlet mass-flow gate failed: check the axis, R0, mesh_units and sector_fraction")

## 14. Solve
Iterates to the residual target, then keeps iterating in batches of 10 until the inlet pressure (= Δp) is flat within `dp_flat_tol` over the last `dp_flat_window` iterations, or the iteration cap. Same rule as Vishesh's pipeline (Decision 3).

In [ ]:
def dp_history():
    hdr, mon = read_monitor_file(MON_FILE)
    return np.array([]) if mon is None else mon[:, hdr.index("p_inlet")]

MAX_IT, W, TOL = cfg["max_iterations"], cfg["dp_flat_window"], cfg["dp_flat_tol"]
t0 = time.time()
s.solution.run_calculation.iterate(iter_count=MAX_IT)
BATCHES = 0
while True:
    p = dp_history()
    DP_SPAN = float(p[-(W + 1):].max() - p[-(W + 1):].min()) if len(p) > W else None
    DP_FLAT = DP_SPAN is not None and DP_SPAN <= TOL
    if DP_FLAT or len(p) >= MAX_IT:
        break
    s.solution.run_calculation.iterate(iter_count=min(10, MAX_IT - len(p)))
    BATCHES += 1
    if len(dp_history()) == len(p):
        print("[warn] Fluent ran no further iterations; leaving the dp-flat loop")
        break
WALL_TIME = round(time.time() - t0, 1)
print(f"iterations {len(p)}; extra batches {BATCHES}; dp span over the last {W}: "
      f"{DP_SPAN if DP_SPAN is None else round(DP_SPAN, 5)} Pa (flat: {DP_FLAT}); wall time {WALL_TIME} s")

## 15. Status
CONVERGED only if every residual is at or below `residual_target` AND Δp is flat (Decision 3). Monitor spread and mass imbalance are reported, not used for the verdict.

In [ ]:
VALS = compute(MON_DEFS)
it, res_data = solver.monitors.get_monitor_set_data(monitor_set_name="residual")
RES_LAST = {k: float(np.asarray(v)[-1]) for k, v in res_data.items()}
HDR, MON = read_monitor_file(MON_FILE)
N_IT = int(len(MON)) if MON is not None else None

SPREAD = {}
if MON is not None and len(MON) > 20:
    w = max(10, int(cfg["stability_window"] * len(MON)))
    for j, nm in enumerate(HDR[1:MON.shape[1]]):
        col = MON[-w:, 1 + j]
        SPREAD[nm] = float((col.max() - col.min()) / max(abs(col[-1]), 1e-30))

mi, mo = VALS["mflow_inlet"], VALS["mflow_outlet"]
IMBALANCE = abs(mi + mo) / abs(mi)
RES_OK = all(v <= cfg["residual_target"] for v in RES_LAST.values())
SPREAD_OK = bool(SPREAD) and all(v <= cfg["stability_tol"] for v in SPREAD.values())
MASS_OK = IMBALANCE <= cfg["mass_imbalance_tol"]
CONVERGED = RES_OK and DP_FLAT
if CONVERGED:
    STATUS = "CONVERGED"
elif RES_OK:
    STATUS = "RESIDUALS MET, DP NOT FLAT"
elif DP_FLAT:
    STATUS = "DP FLAT, RESIDUALS ABOVE TARGET"
else:
    STATUS = "NOT CONVERGED"
print(f"[{TAG}] {STATUS}; iterations {N_IT}; residuals {RES_LAST}")
print(f"  reported: spread ok {SPREAD_OK}; mass imbalance {IMBALANCE:.2e} (ok {MASS_OK})")
print(f"  dp (full) {VALS['p_inlet']:.3f} Pa; vmax {VALS['vmax']:.4f} m/s")

## 15b. Segment Δp
Area-averaged pressure on planes at the two ends of the 25 mm artery segment (`segment_z_start`, `segment_z_start + segment_length`); the same planes as Vishesh's pipeline.

In [ ]:
Z0 = cfg["segment_z_start"]
Z1 = Z0 + cfg["segment_length"]
planes = s.results.surfaces.plane_surface
for name, z in (("seg_up", Z0), ("seg_dn", Z1)):
    planes[name] = {}
    planes[name].method = "xy-plane"
    planes[name].z = z
    rd.surface[f"p_{name}"] = {}
    rd.surface[f"p_{name}"].report_type = "surface-areaavg"
    rd.surface[f"p_{name}"].field = "pressure"
    rd.surface[f"p_{name}"].surface_names = [name]
SEG_P = compute(["p_seg_up", "p_seg_dn"])
DP_SEG = SEG_P["p_seg_up"] - SEG_P["p_seg_dn"]
print(f"segment z = {Z0 * 1e3:.1f} to {Z1 * 1e3:.1f} mm: p_up {SEG_P['p_seg_up']:.3f} Pa, "
      f"p_dn {SEG_P['p_seg_dn']:.3f} Pa, dp_segment {DP_SEG:.3f} Pa")

## 15c. Wall shear stress on the artery wall
Face values with face areas, exported at face centres, then: faces whose centre lies in the segment, area-weighted mean, maximum, and the area fraction below each `low_wss_thresholds` value (Decision 4; Vishesh's method). The first wall zone in `zone_wall` is the artery; any other wall zones (struts) are reported separately.

In [ ]:
WALL_ARTERY = ZONE_WALL[0]

def wall_faces(zone):
    fn = (CASE_DIR / f"{TAG}_faces_{zone}.csv").resolve()
    s.file.export.ascii(file_name=str(fn), surface_name_list=[zone], delimiter="comma",
                        quantities=["wall-shear", "x-wall-shear", "y-wall-shear", "z-wall-shear", "face-area-magnitude"],
                        location="cell-center")
    d = pd.read_csv(fn)
    d.columns = [c.strip() for c in d.columns]
    need = {"z-coordinate", "wall-shear", "face-area-magnitude"}
    if not need <= set(d.columns):
        raise RuntimeError(f"face export is missing {need - set(d.columns)}; it has {list(d.columns)}")
    return d

def wss_stats(d, zlo, zhi):
    m = (d["z-coordinate"] >= zlo) & (d["z-coordinate"] <= zhi)
    w = d.loc[m, "wall-shear"].to_numpy()
    a = d.loc[m, "face-area-magnitude"].to_numpy()
    out = dict(n_faces=int(m.sum()), area_m2=float(a.sum()), wss_mean_pa=float(np.average(w, weights=a)),
               wss_max_pa=float(w.max()), wss_min_pa=float(w.min()))
    for t in cfg["low_wss_thresholds"]:
        out[f"low_wss_frac_lt_{t:g}"] = float(a[w < t].sum() / a.sum())
    return out

FACES = {z: wall_faces(z) for z in ZONE_WALL}
WSS_SEG = wss_stats(FACES[WALL_ARTERY], Z0, Z1)
WSS_WALL = wss_stats(FACES[WALL_ARTERY], -np.inf, np.inf)
WSS_STRUT = {z: wss_stats(d, -np.inf, np.inf) for z, d in FACES.items() if z != WALL_ARTERY}

# cross-check: Fluent's own area-weighted average over the same wall should match the whole-wall mean
if len(ZONE_WALL) == 1:
    diff = abs(WSS_WALL["wss_mean_pa"] - VALS["wss_wall_areaavg"]) / VALS["wss_wall_areaavg"]
    print(f"[check] whole-wall mean {WSS_WALL['wss_mean_pa']:.5f} Pa vs Fluent area-average "
          f"{VALS['wss_wall_areaavg']:.5f} Pa ({diff:.3%})")
    if diff > 0.005:
        print("[warn] face export and Fluent's report differ by more than 0.5 %: check the export")
pd.DataFrame({"segment (artery)": WSS_SEG, "whole artery wall": WSS_WALL, **WSS_STRUT})

## 16. Write case/data, exports and summary
Node exports for plots and post-processing; the segment WSS numbers come from cell 15c.

In [ ]:
s.file.write(file_type="case-data", file_name=str((CASE_DIR / f"{TAG}.cas.h5").resolve()))

base_q = ["pressure", "x-velocity", "y-velocity", "z-velocity", "velocity-magnitude",
          "strain-rate-mag", "viscosity-lam"]
turb_q = ["turb-kinetic-energy", "specific-diss-rate", "viscosity-turb"] if cfg["viscous_model"] == "sst" else []
wall_q = ["pressure", "wall-shear", "x-wall-shear", "y-wall-shear", "z-wall-shear",
          "strain-rate-mag", "viscosity-lam"]
jobs = [("wall", ZONE_WALL, wall_q)]
if ZONE_SYM:
    jobs.append(("symmetry", ZONE_SYM, base_q + turb_q))
EXPORTS = {}
for label, surfaces, quantities in jobs:
    fn = (CASE_DIR / f"{TAG}_{label}.csv").resolve()
    s.file.export.ascii(file_name=str(fn), surface_name_list=list(surfaces), delimiter="comma",
                        quantities=quantities, location="node")     # 26.1 argument names (Task 0.3, 2 Oct)
    EXPORTS[label] = str(fn)

RECORD = dict(
    tag=TAG, created=datetime.datetime.now().isoformat(timespec="seconds"),
    fluent=solver.get_fluent_version(), procs=PROCS, mesh=str(MESH_PATH.resolve()), setup_csv=SETUP_CSV,
    inlet_u=INLET_U, inlet_profile=cfg["inlet_profile"], inlet_expression=INLET_EXPR,
    viscous_model=cfg["viscous_model"], **FLOW,
    A_inlet=A_IN, A_inlet_ratio=A_IN / FLOW["A_inlet_nominal_m2"], R_inlet=R_IN,
    mdot_target=MDOT_TARGET, mdot_init=MDOT_INIT,
    iterations=N_IT, extra_batches=BATCHES, final_residuals=RES_LAST, residuals_met=RES_OK,
    dp_flat=DP_FLAT, dp_span_last_window_pa=DP_SPAN, converged=CONVERGED,
    monitor_spread=SPREAD, spread_ok=SPREAD_OK, mass_imbalance=IMBALANCE, mass_ok=MASS_OK,
    dp_full_pa=VALS["p_inlet"], segment_z_m=[Z0, Z1], p_seg_up_pa=SEG_P["p_seg_up"], p_seg_dn_pa=SEG_P["p_seg_dn"],
    dp_segment25_pa=DP_SEG, wss_wall_areaavg_pa=VALS["wss_wall_areaavg"], wss_wall_max_pa=VALS["wss_wall_max"],
    wss_segment=WSS_SEG, wss_whole_artery_wall=WSS_WALL, wss_strut_walls=WSS_STRUT, vmax_mps=VALS["vmax"],
    tawss="NOT MEASURED (Phase 3)", osi="NOT MEASURED (Phase 3)",
    mesh_from=MESH_FROM, mesh_report=(MESH_REPORT if MESH_FROM == "step" else None),
    status=STATUS, wall_time_s=WALL_TIME, exports=EXPORTS,
)
(CASE_DIR / f"{TAG}_summary.json").write_text(json.dumps(RECORD, indent=2,
    default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o)))   # FluentVersion -> text

summary_csv = out / "run_summary.csv"
cols = ["tag", "inlet_u", "viscous_model", "iterations", "status", "converged", "mass_imbalance",
        "dp_full_pa", "dp_segment25_pa", "vmax_mps", "A_inlet_ratio", "wall_time_s"]
row = pd.DataFrame([{**{c: RECORD[c] for c in cols},
                     "wss_seg_mean_pa": WSS_SEG["wss_mean_pa"], "wss_seg_max_pa": WSS_SEG["wss_max_pa"],
                     **{f"seg_{k}": v for k, v in WSS_SEG.items() if k.startswith("low_wss")}}])
row.to_csv(summary_csv, mode="a", header=not summary_csv.exists(), index=False)
print("written:", CASE_DIR)

## 17. Plots
Monitors should be flat at the end; the Carreau points should sit on the curve; WSS along the vessel.

In [ ]:
if MON is not None:
    names = HDR[1:MON.shape[1]]
    fig, axs = plt.subplots(1, len(names), figsize=(2.9 * len(names), 2.5))
    for ax, j, nm in zip(np.atleast_1d(axs), range(1, MON.shape[1]), names):
        ax.plot(MON[:, 0], MON[:, j], lw=1)
        ax.set_title(nm, fontsize=8); ax.set_xlabel("iteration", fontsize=8); ax.tick_params(labelsize=7)
    fig.suptitle(f"{TAG}: {STATUS}", fontsize=9); plt.tight_layout(); plt.show()

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
if "symmetry" in EXPORTS and cfg["viscosity_model"] == "carreau":
    d = pd.read_csv(EXPORTS["symmetry"]); d.columns = [c.strip() for c in d.columns]
    g = d["strain-rate-mag"].to_numpy()
    gg = np.logspace(np.log10(max(g[g > 0].min(), 1e-3)), np.log10(g.max()), 200)
    axs[0].loglog(g, d["viscosity-lam"], ".", ms=2, alpha=0.4, label="Fluent (symmetry nodes)")
    axs[0].loglog(gg, carreau(gg), "k-", lw=1.2, label="Carreau, CSV constants")
    axs[0].set_xlabel("strain rate [1/s]"); axs[0].set_ylabel("viscosity [Pa s]"); axs[0].legend(fontsize=8)
d = pd.read_csv(EXPORTS["wall"]); d.columns = [c.strip() for c in d.columns]
axs[1].plot(d["z-coordinate"] * 1e3, d["wall-shear"], ".", ms=2, alpha=0.4)
axs[1].set_xlabel("axial position z [mm]"); axs[1].set_ylabel("WSS [Pa]")
fig.suptitle(TAG, fontsize=9); plt.tight_layout(); plt.show()

## 17b. Compare with Vishesh's pipeline (Phase 1)
Picks the `L7_cfd_qoi.csv` row for `QOI_CASE` and `INLET_U`. Pass marks (Decision 5): full Δp within 1 %, segment mean WSS within 2 %. The rest is reported. The 08_cfd cases are from 27 Sep, before the 28 Sep CFD fixes: a miss may be a settings difference, not an error.

In [ ]:
qoi = pd.read_csv(QOI_FILE)
QKEY = f"{QOI_CASE}_U{INLET_U:.2f}".replace(".", "p")
match = qoi[qoi["case"] == QKEY]
if match.empty:
    raise ValueError(f"{QKEY} is not in {QOI_FILE}; cases there: {list(qoi['case'])}")
q = match.iloc[0]

def rel(a, b):
    return (a - b) / b if b else float("nan")

rows = [
    ("dp full domain [Pa]", VALS["p_inlet"], q["pressure_drop_pa"], 0.01),
    ("WSS mean, segment [Pa]", WSS_SEG["wss_mean_pa"], q["wss_area_mean_pa"], 0.02),
    ("WSS max, segment [Pa]", WSS_SEG["wss_max_pa"], q["wss_max_pa"], None),
    ("WSS mean, whole wall [Pa]", WSS_WALL["wss_mean_pa"], q["wss_area_mean_whole_wall_pa"], None),
]
for t, col in ((0.5, "low_wss_area_frac_lt_0.5pa"), (0.4, "low_wss_area_frac_lt_0.4pa"), (1.0, "low_wss_area_frac_lt_1.0pa")):
    rows.append((f"low-WSS area fraction < {t:g} Pa", WSS_SEG[f"low_wss_frac_lt_{t:g}"], q[col], None))
rows.append(("iterations", N_IT, q["iterations"], None))
if MESH_FROM == "step":
    rows.append(("cells", MESH_REPORT["cell_count"], q["cells"], None))
    rows.append(("min orthogonal quality", MESH_REPORT["min_orthogonal_quality"], q["min_orthogonal_quality"], None))

table = []
for name, ours, theirs, tol in rows:
    if name.startswith("low-WSS"):
        diff = f"{(ours - theirs) * 100:+.2f} points"
    elif name in ("iterations", "cells"):
        diff = f"{int(ours) - int(theirs):+d}"
    else:
        diff = f"{rel(ours, theirs):+.2%}"
    result = "" if tol is None else ("PASS" if abs(rel(ours, theirs)) <= tol else "FAIL")
    table.append(dict(quantity=name, notebook=ours, pipeline_27Sep=theirs, difference=diff,
                      pass_mark="" if tol is None else f"{tol:.0%}", result=result))
COMPARE = pd.DataFrame(table)
COMPARE.insert(0, "case", QKEY)
COMPARE.to_csv(CASE_DIR / f"{TAG}_vs_qoi.csv", index=False)
cmp_all = out / "phase1_comparison.csv"
COMPARE.to_csv(cmp_all, mode="a", header=not cmp_all.exists(), index=False)
print(f"{QKEY}: notebook {STATUS} in {N_IT} iterations; pipeline converged = {q['converged']} in {q['iterations']}")
COMPARE

## 18. Close Fluent
Run when you have finished both velocities.

In [ ]:
solver.exit()

## Output layout
```
artery_runs_v3/
  run_summary.csv                  one row per run (appended)
  <mesh>_<model>_U0p10/            one folder per run
      case_setup_used.csv          exact CSV used
      *.cas.h5 / *.dat.h5          Fluent case and data
      *_monitors.out               monitor history
      *_wall.csv                   wall nodes: pressure, WSS and components, viscosity
      *_symmetry.csv               symmetry-plane nodes: p, velocity, strain rate, viscosity
      *_summary.json               full record
```